# Run all notebooks in the project

In [7]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Movement smoothness as a subclinical marker in low back pain - Segmentation and SPARC/NNP calculation"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Ancelin Gely (and Copilot)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"by {authors}"))

# Movement smoothness as a subclinical marker in low back pain - Segmentation and SPARC/NNP calculation

05 May 2026, 10:29:04

by Ancelin Gely (and Copilot)

# Scientific problem

## Context
Low back pain represents the leading cause of disability worldwide. At the individual level, low back pain has substantial consequences, including activity limitations (1) as well as spinal kinematic alterations (2).

Movement smoothness has a potential to provide valuable information about sensorimotor control and to support patient assessment.it reflects the degree to which a movement is performed continuously without interruption, independently of its amplitude or duration(3).

Several methods exist to evaluate movement smoothness, including the spectral arc lenght (SPARC), the log dimensionless jerk (LDJ), the normalized number of peaks (NPP). The SPARC method has been shown to be more valid, sensitive and robust than the other methods. NNP as the advantage of being less computationally intensive and easier to implement in clinical settings. Moreover as the movement duration in our study is relatively short, the NNP method might be less affected by movement duration than the SPARC method (3,4,5). In a previous study, we have shown that SPARC and NNP were significantly correlated and can be interchangeable in a set of 10 low back pain patients performing a flexion-extension task (6).

In the present study, we evaluation movement smoothness during 5 repetions of flexion extension. However, we don't know if the SPARC and NNP give similar result when we calculate them for each repetition of the movement or when we calculate them for the whole movement. This is important to know because in clinical practice, it is more common to evaluate the whole movement rather than segmenting it into repetitions

## Aim 
The aime of this study is to investigate the relationship between the whole signal and the segmented signal for both SPARC and NNP methods in individuals with low back pain. These results will help us to determine if the SPARC and NNP metrics calculated for the whole movement can be used as a proxy for the metrics calculated for each repetition, which would simplify the assessment of movement smoothness in clinical practice.

## Methodology

## Participants
10 individuals with low back pain participated in this study. The target population consisted of adults aged between 18 to 65 years with chronic low back pain persisting for more than three months. Participants were eligible for inclusion if they had a body mass index between 18 and 30 kg/m^2 and were undergoing rehabilitation in the physical and rehabilitation medicine department of Montpellier University Hospital

## Outcomes
Movement smoothness was evaluated based on the velocity profile given by the gyroscope.

SPARC formula : SPARC = - ∫ |dV(ω)/dω| dω Where V(ω) is the Fourier magnitude spectrum of the velocity profile and ω is the angular frequency.

NNP formula : NNP = (number of peaks in the velocity profile) / (total duration of the movement).

# Aim of the code 

The aim of this code is to extract the velocity profile from the gyroscope data, segment the movement into repetition of flexion - extension and compute the SPARC and NNP for each repetition and for the whol movement. 

# Data organization

## File name 
Flexion_Dos_OOX_av.xlsx
X is the patient ID (1-10)

## File structure
The data is organized as a table with different sheets : 
- general informations 
- Markers
- Segment orientation - Quat
- Segment orientation - Euler
- Segment position 
- Segment velocity
- Segment acceleration
- Segment angular velocity
- Ergonomic joint angle 
- Center of mass
- Sensor free acceleration
- Sensor magnetic field
- Sensor orientation - Quat
- Sensor orientation - Euler

## Gyroscope data
The gyroscope data is located in the sheet "Segment angular velocity". It contains the angular velocity of the segment in three dimensions (x, y, z). 

Columns :
- Frame	
- Pelvis x, y, z		
- L5 x, y, z
- L3 x, y, z
- T12 x, y, z	
- T8 x, y, z

# Notebook organization

In this jupiter noebook, we will perform the following steps :
1. Load the data and filter the gyroscope data for L3 Y (L3)
2. Segment the movement into repetitions of flexion-extension
3. Compute the SPARC and NNP for each repetition and for the whole movement
4. Add the results to a dataframe and save it as a xlsx file




# Code 

## Import libraries



In [8]:
# If you need to install the libraries, uncomment the following lines and run them once. After that, you can comment them again.
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install scipy
# !pip install IPython

# Import library
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import find_peaks

## Function definitions 

The 4 steps (load and filter the gyroscope data, segment the movement, compute the SPARC and NNP, and save the results) of the code are defined as functions "analyse_signal" to make the code more organized and reusable.

It uses the following functions :
- butterworth_filter : to filter the signal with a butterworth filter
- find_peaks : to find the peaks in the velocity profile
- compute_SPARC : to compute the SPARC of the signal
- fft : to compute the Fourier transform of the signal




In [ ]:
# Signal processing parameters
fs = 60            # Hz
fc = 10            # low-pass cutoff (Hz)
eps = 0.01         # noise threshold (deg/s)
min_peak_distance = 0.5  # seconds
dt = 1 / fs         # time step (s)

# Function to analyze the signal and compute SPARC for each segment
def analyze_signal(file_path, sheet_name, column_name):

    #######################################
    ### Load data and filter the signal ###
    #######################################
    signal = pd.read_excel(file_path, sheet_name=sheet_name)

    if "Time (s)" not in signal.columns:
        if "Frame" not in signal.columns:
            raise ValueError("Time (s) or Frame column missing")
        signal["Time (s)"] = signal["Frame"] / fs

    omega = signal[column_name].to_numpy()

    """
    The signal is by frame and we want it by time. 
    We have to create a new time vector based on the Frame column.
    time = signal["Frame"] / fs
    """

    # Filter signal
    b, a = butter(2, fc / (fs / 2), btype="low")
    omega_filt = filtfilt(b, a, omega)

    """
    A low pass filter is applied to the signal to remove high frequency noise. 
    2 : The order of the filter is set to 2, which provides a good balance between filtering and preserving the shape of the signal.
    fc : The cutoff frequency is set to 10 Hz, which is a common choice for human movement data. 
    fs/2 : The cutoff frequency is divided by the Nyquist frequency (fs/2) to normalize it for the Butterworth filter design.
    btype ="low" : The type of the filter is set to "low" to keep the low-frequency components.
    """

    #########################################
    ########## Segment detection ############
    #########################################

    # negative segment
    neg_mask = omega_filt < -eps
    neg_transitions = np.diff(neg_mask.astype(int))
    neg_starts = np.where(neg_transitions == 1)[0] + 1
    neg_ends = np.where(neg_transitions == -1)[0] + 1
    if neg_mask[0]:
        neg_starts = np.insert(neg_starts, 0, 0)
    if neg_mask[-1]:
        neg_ends = np.append(neg_ends, len(neg_mask))

    # Positive segment
    pos_mask = omega_filt > eps
    pos_transitions = np.diff(pos_mask.astype(int))
    pos_starts = np.where(pos_transitions == 1)[0] + 1
    pos_ends = np.where(pos_transitions == -1)[0] + 1
    if pos_mask[0]:
        pos_starts = np.insert(pos_starts, 0, 0)
    if pos_mask[-1]:
        pos_ends = np.append(pos_ends, len(pos_mask))

    # Peak detection
    positive_peaks, _ = find_peaks(
        omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    negative_peaks, _ = find_peaks(
        -omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    # Associate segments to peaks
    negative_segments = []
    for p in negative_peaks:
        idx = np.where((neg_starts <= p) & (neg_ends >= p))[0]
        if len(idx) == 1:
            negative_segments.append((int(neg_starts[idx[0]]), int(neg_ends[idx[0]])))

    positive_segments = []
    for p in positive_peaks:
        idx = np.where((pos_starts <= p) & (pos_ends >= p))[0]
        if len(idx) == 1:
            positive_segments.append((int(pos_starts[idx[0]]), int(pos_ends[idx[0]])))

    # Suppress the segment if it is less than 0.5 second long
    negative_segments = [
        (s, e) for s, e in negative_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    positive_segments = [
        (s, e) for s, e in positive_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    # suppress the segment if it is the same sign as the previous one.
    # If a segment with a different sign is between them, keep it.
    def suppress_consecutive_segments(segments):
        if not segments:
            return []
        suppressed = [segments[0]]
        for s, e in segments[1:]:
            last_s, last_e = suppressed[-1]
            if (s - last_e) > 1:
                suppressed.append((s, e))
        return suppressed

    negative_segments = suppress_consecutive_segments(negative_segments)
    positive_segments = suppress_consecutive_segments(positive_segments)


    ####################################
    ########## Compute SPARC  ##########
    #################################### 
    def compute_sparc(signal):

        """
        Parameters
        ----------
        signal : array-like
            The input signal for which to compute the SPARC metric.

        Returns
        -------
        float
            The computed SPARC metric.
        """

        signal = signal - np.mean(signal)
        n = len(signal)

        spectrum = np.abs(np.fft.rfft(signal))
        spectrum = spectrum / np.max(spectrum)
        spectrum += 1e-10

        #freqs = np.linspace(0, 1, len(spectrum))
        freqs = np.fft.rfftfreq(n, d=1/fs)

        threshold = 0.05
        valid = spectrum > threshold
        
        spectrum = spectrum[valid]
        freqs = freqs[valid]

        log_spectrum = np.log(spectrum)

        df = np.diff(freqs)
        ds = np.diff(log_spectrum)

        arc_length = np.sum(np.sqrt(df**2 + ds**2))
        return -arc_length

    # SPARC for each segment
    positive_sparc = []
    for start, end in positive_segments:
        positive_sparc.append(compute_sparc(omega_filt[start:end]))

    negative_sparc = []
    for start, end in negative_segments:
        negative_sparc.append(compute_sparc(omega_filt[start:end]))

    # SPARC for the whole signal
    SPARC = compute_sparc(omega_filt)


    ########################################
    ############# Compute NNP###############
    ########################################
    extension_peak_counts = []
    for s, e in negative_segments:
        segment_time = signal["Time (s)"].iloc[s:e] - signal["Time (s)"].iloc[s]
        segment_omega = omega_filt[s:e]

        #peaks, _ = find_peaks(segment_omega, prominence=0.005)
        #peaks, _ = find_peaks(segment_omega, prominence=0.003)
        peaks, _ = find_peaks(segment_omega, prominence=0.01)

        peak_count = len(peaks)
        duration = segment_time.iloc[-1] - segment_time.iloc[0]

        normalized_peak_count = peak_count / duration if duration > 0 else 0
        extension_peak_counts.append(normalized_peak_count)

    flexion_peak_counts = []
    for s, e in positive_segments:
        segment_time = signal["Time (s)"].iloc[s:e] - signal["Time (s)"].iloc[s]
        segment_omega = omega_filt[s:e]

        #peaks, _ = find_peaks(segment_omega, prominence=0.005)
        #peaks, _ = find_peaks(segment_omega, prominence=0.003)
        peaks, _ = find_peaks(segment_omega, prominence=0.01)
        
        peak_count = len(peaks)
        duration = segment_time.iloc[-1] - segment_time.iloc[0]

        normalized_peak_count = peak_count / duration if duration > 0 else 0
        flexion_peak_counts.append(normalized_peak_count)

    # NNP for the whole signal
    peaks, _ = find_peaks(omega_filt, prominence = 0.005)
    # Normalized number of peaks (NNP) for the whole signal
    nnp = len(peaks) / (signal["Time (s)"].iloc[-1] - signal["Time (s)"].iloc[0])


    ##################################
    ###### Display results ###########
    ##################################

    # I have to analyse 10 signal and I want all result in a
    # Dataframe for each repetition.
    global results_repetition
    if "results_repetition" not in globals() or results_repetition is None:
        results_repetition = pd.DataFrame(
            columns=["Signal name", "SPARC_flexion", "SPARC_extension", "NNP_flexion", "NNP_extension"]
        )
    
    signal_name = file_path.split("/")[-1].split(".")[0]


    # Be sure that "All arrays must be of the same length" does not happen when I concatenate the new row to the results_repetition dataframe.
    if len(positive_sparc) != len(negative_sparc) or len(positive_sparc) != len(flexion_peak_counts) or len(positive_sparc) != len(extension_peak_counts):
        max_len = max(len(positive_sparc), len(negative_sparc), len(flexion_peak_counts), len(extension_peak_counts))
        positive_sparc += [np.nan] * (max_len - len(positive_sparc))
        negative_sparc += [np.nan] * (max_len - len(negative_sparc))
        flexion_peak_counts += [np.nan] * (max_len - len(flexion_peak_counts))
        extension_peak_counts += [np.nan] * (max_len - len(extension_peak_counts))
        
    new_row = pd.DataFrame({
        "Signal name": [f"{signal_name}_rep{i+1}" for i in range(len(positive_sparc))],
        "SPARC_flexion": positive_sparc,
        "SPARC_extension": negative_sparc,
        "NNP_flexion": flexion_peak_counts,
        "NNP_extension": extension_peak_counts
    })


    results_repetition = pd.concat([results_repetition, new_row], ignore_index=True)
    results_repetition = results_repetition.drop_duplicates(subset=["Signal name"], keep="last").reset_index(drop=True)

    # Dataframe for the whole signal.
    global results
    if "results" not in globals() or results is None:
        results = pd.DataFrame(
            columns=["Signal name", "SPARC", "NNP"]
        )

    new_row_2 = pd.DataFrame({
        "Signal name": [signal_name],
        "SPARC": [SPARC],
        "NNP": [nnp]
    })

    results = pd.concat([results, new_row_2], ignore_index=True)
    results = results.drop_duplicates(subset=["Signal name"], keep="last").reset_index(drop=True)

    # Merged results 
    results_repetition['NNP'] = results_repetition['Signal name'].str.extract(r'(FlexionDos_\d+_av)')[0].map(results.set_index('Signal name')['NNP'])
    results_repetition['SPARC'] = results_repetition['Signal name'].str.extract(r'(FlexionDos_\d+_av)')[0].map(results.set_index('Signal name')['SPARC'])

    results_repetition.to_excel("results/results_repetition.xlsx", index=False)
    return results_repetition

In [10]:
# Patient : 1
FILE_PATH = "data/FlexionDos_001_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 2
FILE_PATH = "data/FlexionDos_002_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 3
FILE_PATH = "data/FlexionDos_003_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 4
FILE_PATH = "data/FlexionDos_004_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 5
FILE_PATH = "data/FlexionDos_005_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 6
FILE_PATH = "data/FlexionDos_006_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 7
FILE_PATH = "data/FlexionDos_007_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 8
FILE_PATH = "data/FlexionDos_008_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# # Patient : 9
FILE_PATH = "data/FlexionDos_009_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

# Patient : 10
FILE_PATH = "data/FlexionDos_010_av.xlsx"
SHEET_NAME = "Segment Angular Velocity"
COLUMN_NAME = "L3 y"

analyze_signal(FILE_PATH, SHEET_NAME, COLUMN_NAME)

,Signal name,SPARC_flexion,SPARC_extension,NNP_flexion,NNP_extension,NNP,SPARC
0,FlexionDos_001_av_rep1,-6.511600,-3.649579,1.186441,0.692308,1.670103,-6.874907
1,FlexionDos_001_av_rep2,-3.310238,-3.073053,1.495327,1.276596,1.670103,-6.874907
2,FlexionDos_001_av_rep3,-4.599152,-3.756300,1.114551,0.833333,1.670103,-6.874907
3,FlexionDos_001_av_rep4,-2.867815,-4.233747,0.919540,0.566038,1.670103,-6.874907
4,FlexionDos_001_av_rep5,-3.507003,-3.619190,1.660900,0.818182,1.670103,-6.874907
5,FlexionDos_002_av_rep1,-5.608169,-4.068137,2.967033,4.903846,3.946588,-6.735183
6,FlexionDos_002_av_rep2,-5.447141,-9.837785,3.483871,3.139535,3.946588,-6.735183
7,FlexionDos_002_av_rep3,-4.273620,-4.128918,2.441860,3.545455,3.946588,-6.735183
8,FlexionDos_002_av_rep4,-5.333908,-4.183752,3.208556,3.367347,3.946588,-6.735183
9,FlexionDos_002_av_rep5,-6.085372,-8.834924,3.103448,4.309392,3.946588,-6.735183
